In [1]:
with open('../data/langv1.txt', 'r') as f:
    text = f.read()

In [2]:
len(text)

130399

In [3]:
print(text[:1000])

fira obama bhago
fira obama yumyum
fira obama nini
fira obama toing
fira obama snan
fira obama icarus
fira obama jojo
fira obama perv
fira obama sisy
fira obama bobder
fira obama fenu
fira obama zari
fira obama loti
fira obama rime
fira obama paku
fira obama soro
fira obama davi
fira obama nexo
fira obama naku
fira obama savi
fira gugugaga bhago
fira gugugaga yumyum
fira gugugaga nini
fira gugugaga toing
fira gugugaga snan
fira gugugaga icarus
fira gugugaga jojo
fira gugugaga perv
fira gugugaga sisy
fira gugugaga bobder
fira gugugaga fenu
fira gugugaga zari
fira gugugaga loti
fira gugugaga rime
fira gugugaga paku
fira gugugaga soro
fira gugugaga davi
fira gugugaga nexo
fira gugugaga naku
fira gugugaga savi
fira osama bhago
fira osama yumyum
fira osama nini
fira osama toing
fira osama snan
fira osama icarus
fira osama jojo
fira osama perv
fira osama sisy
fira osama bobder
fira osama fenu
fira osama zari
fira osama loti
fira osama rime
fira osama paku
fira osama soro
fira osama davi
fira

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 abcdefghijklmnoprstuvwxyz
27


In [5]:
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode('hola senor'))
print(decode(encode('hola senor')))

[9, 16, 13, 2, 1, 19, 6, 15, 16, 18]
hola senor


In [6]:
import torch
data = torch.tensor(encode(text))
print(data.shape, data.dtype)
print(data[:100])

torch.Size([130399]) torch.int64
tensor([ 7, 10, 18,  2,  1, 16,  3,  2, 14,  2,  1,  3,  9,  2,  8, 16,  0,  7,
        10, 18,  2,  1, 16,  3,  2, 14,  2,  1, 25, 21, 14, 25, 21, 14,  0,  7,
        10, 18,  2,  1, 16,  3,  2, 14,  2,  1, 15, 10, 15, 10,  0,  7, 10, 18,
         2,  1, 16,  3,  2, 14,  2,  1, 20, 16, 10, 15,  8,  0,  7, 10, 18,  2,
         1, 16,  3,  2, 14,  2,  1, 19, 15,  2, 15,  0,  7, 10, 18,  2,  1, 16,
         3,  2, 14,  2,  1, 10,  4,  2, 18, 21])


In [7]:
n = int(len(data) * 0.8)
train_data = data[:n]
val_data = data[n:]

In [8]:
len(val_data), len(train_data)

(26080, 104319)

In [9]:
block_size = 8
train_data[:block_size+1]

tensor([ 7, 10, 18,  2,  1, 16,  3,  2, 14])

In [10]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]

    print(f"when input is {context}, the target is {target}")

when input is tensor([7]), the target is 10
when input is tensor([ 7, 10]), the target is 18
when input is tensor([ 7, 10, 18]), the target is 2
when input is tensor([ 7, 10, 18,  2]), the target is 1
when input is tensor([ 7, 10, 18,  2,  1]), the target is 16
when input is tensor([ 7, 10, 18,  2,  1, 16]), the target is 3
when input is tensor([ 7, 10, 18,  2,  1, 16,  3]), the target is 2
when input is tensor([ 7, 10, 18,  2,  1, 16,  3,  2]), the target is 14


In [11]:
torch.manual_seed(6767)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])  #torch.stack just turns it into a tensor in this case.
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch("train")
print(xb.shape)
print(xb)

print(yb.shape)
print(yb)

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]

        print(f"when za context is {context.tolist()}, the target is {target}")

torch.Size([4, 8])
tensor([[22,  0, 18, 16, 15, 10,  1,  5],
        [ 2,  7, 12,  2,  1, 17,  2, 12],
        [21, 12, 21,  1,  3, 16,  3,  5],
        [15,  6, 14, 16,  1,  5,  2,  8]])
torch.Size([4, 8])
tensor([[ 0, 18, 16, 15, 10,  1,  5, 21],
        [ 7, 12,  2,  1, 17,  2, 12, 21],
        [12, 21,  1,  3, 16,  3,  5,  6],
        [ 6, 14, 16,  1,  5,  2,  8, 10]])
when za context is [22], the target is 0
when za context is [22, 0], the target is 18
when za context is [22, 0, 18], the target is 16
when za context is [22, 0, 18, 16], the target is 15
when za context is [22, 0, 18, 16, 15], the target is 10
when za context is [22, 0, 18, 16, 15, 10], the target is 1
when za context is [22, 0, 18, 16, 15, 10, 1], the target is 5
when za context is [22, 0, 18, 16, 15, 10, 1, 5], the target is 21
when za context is [2], the target is 7
when za context is [2, 7], the target is 12
when za context is [2, 7, 12], the target is 2
when za context is [2, 7, 12, 2], the target is 1
when za 

In [12]:
from torch import nn
from torch.nn import functional as F
torch.manual_seed(6767)

class BigramModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding(idx)

        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim = -1)

            idx_next = torch.multinomial(probs, num_samples =1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
model = BigramModel(vocab_size)

logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

decode(model.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_tokens = 50)[0].tolist())

torch.Size([32, 27])
tensor(3.6173, grad_fn=<NllLossBackward0>)


'\nrpdvrhktdvwpzzwkzyxsccdupgfmnmzzb\nihcgexm kz\namfti'

In [13]:
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-3)

In [14]:
batch_size = 32
for steps in range(10000):
    xb, yb= get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad()

    loss.backward()
    optimizer.step()

print(loss.item())
print(logits.shape)


1.9353529214859009
torch.Size([256, 27])


In [15]:
print(vocab_size)

27


In [16]:
print(decode(model.generate(idx = torch.zeros((1,1), dtype = torch.long), max_new_tokens=400)[0].tolist()))


leni namo rica dexoku iniobdra na vo
hichagakukadag
bo xo ro ticha nu drojaku lnen
s
li cr
rv
ha dawkuyr
fena mosobo
ti
n
numa sto kumagi ku kenu
mdakumi ri
nelu sawkaka numamimir lorotos
s
jobda pyradu kuginuma sa ghicarireni yuku du m
zeroi nustomy
denir li se
te feku pi nem nemdu zemy
bhi vi da kav
bada ga sa na kugapa fiso gu ku ba badumir gu davi
li
ren
zechagicawkumaku du
ti lora renarv
joni


In [17]:
B, T, C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [18]:
xbow = torch.zeros((B,T,C)) # bow is bag of words

for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev, 0)



In [19]:
# doing the same thing as the upper cell, just way more efficient
wei  = torch.tril(torch.ones(T,T))
wei = wei / torch.sum(wei, 1, keepdim=True)
xbow2 = wei @ x #  the b here is auto-summoned by pytorch in wei(B,T,T) @ (B, T, C) -> (B,T,C)
torch.allclose(xbow, xbow2)

True

In [20]:
# another cool shitt
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros(T,T)
wei = wei.masked_fill(tril == 0, float('-inf')) #stuff i learned from 3blue1brown
wei = F.softmax(wei, dim=1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

True

we do the above stuff, so that the word/char at a position t cant have access to information of the words/chars ahead of that particular word/char, only having access to the informatin precedding t

In [21]:
torch.manual_seed(6768)
a = torch.tril(torch.ones((3,3)))
# tril is a crazy cool func if you really think about it -- pretty useful when you matmul using it
a = a / torch.sum(a, 1, keepdim=True) # using it to avg in an incremental way
b = torch.randint(0,10, (3,2)).float()
c = a @ b
print(a)
print(b)
print(c)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[5., 7.],
        [1., 1.],
        [2., 5.]])
tensor([[5.0000, 7.0000],
        [3.0000, 4.0000],
        [2.6667, 4.3333]])


In [22]:
torch.manual_seed(6767)
B, T, C = 4, 8, 32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x) # (B,T,16)
q = query(x) # (B,T,16)

#now focuses on a token based on learned params instead of giving equal importance to each token
wei = q @ k.transpose(-2, -1)  # (B, T, 16) @ (B, 16 , T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float('-inf')) # no peeking foward hehe
wei = F.softmax(wei, dim=1)
# out = wei @ x

v = value(x)
out = wei @ v # the v gets aggregated
out.shape

torch.Size([4, 8, 16])

In [26]:
#all the em-dashes are user generated.

import torch
from torch import nn
from torch.nn import functional as F

batch_size = 64
block_size = 256 #context length
max_iters = 5000
eval_interval = 500
lr = 3e-4
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 200
n_embed = 384
n_head = 6
n_layers = 6
dropout = .2
# head_size = 16

# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('../data/langv1.txt', 'r', encoding = 'utf-8') as f:
    text = f.read()

torch.manual_seed(1337)


chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text))



n = int(len(data) * 0.9)
train_data = data[:n]
val_data = data[n:]


def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])  #torch.stack just turns it into a tensor in this case.
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch("train")


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x) # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)

        #now focuses on a token based on learned params instead of giving equal importance to each token
        wei = (q @ k.transpose(-2, -1)) * (self.head_size ** -0.5)  # (B, T, head_size) @ (B, head_size , T) ---> (B, T, T)


        # wei = torch.zeros(T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0 , float('-inf')) # no peeking foward hehe, :T implemeted cause at the time of generation the T might be diff when starting
        wei = F.softmax(wei, dim=-1)  # the -inf elems gets the prob of 0 when passed thorugh softmax
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v # the v gets aggregated
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embed) # num_heads * head_size = n_embed. so, its 32 -> 32
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1) #combining all the features to form 32 features again.
        out = self.proj(out)
        return self.dropout(out)

class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed), #basicaly projection for ffwd, coupled straight into the sequential.
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embed, n_head):
        super().__init__()
        self.sa_head = MultiHeadAttention(n_head, n_embed//n_head)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        # adding residuals
        x = x + self.sa_head(self.ln1(x)) # we fork off, compute the sa and come back to the original pathway.
        x = x + self.ffwd(self.ln2(x))
        # we feed the x into LayerNorm before sending it to sa or ffwd.
        return x

class BigramModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.positional_embedding = nn.Embedding(block_size, n_embed)
        # self.sa_head = Head(n_embed)
        # self.sa_head = MultiHeadAttention(4, n_embed // 4)
        self.blocks = nn.Sequential(
            *[Block(n_embed, n_head) for _ in range(n_layers)]
        )
        self.ln = nn.LayerNorm(n_embed)
        # self.ffwd = FeedForward(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.positional_embedding(torch.arange(T, device = device ))
        x = tok_emb + pos_emb
        # x = self.sa_head(x)
        # x = self.ffwd(x)
        x = self.blocks(x)
        x = self.ln(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None

        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) #converting the shape casue the cross_entropy doenst take 3 dims, so we're combining all the elements from diff batches into a single vector dim.
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        #!!!HEAVY LEARNING FROM HEAVY MISTAKE
        # idx_cropped = idx[:, -block_size:] -----> lol i tried to bring it up here thinking i was doing some crazy optimization -- welp, i was sending the autoregressive mech. on fire-- basically if it was out of the loop, it wont be able to use the newly generated tokens appended to it inside the loop -- so for every iter it would have just taken 0 as input basically "\n" xD
        for _ in range(max_new_tokens):
            idx_cropped = idx[:, -block_size:] #we have to crop it to only have 8 char because we have positional encoding
            #it will only have the last 8 elems/chars

            logits, loss = self(idx_cropped)
            logits = logits[:, -1, :] #taking only the last token, cause only the last token is used to generate the next token

            probs = F.softmax(logits, dim = -1)
             # will do softmax for each batch item -> here there is no sense of batch because the batch_size will be 1 but there is max_new_tokens. so, it would generate (1,vocab_size) softmax prob dist for every iter(basically for every new token)

            idx_next = torch.multinomial(probs, num_samples =1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
model = BigramModel()
model = model.to(device)


optimizer = torch.optim.AdamW(model.parameters(), lr = lr) # AdamW is just adam-chan with weight decay decoupled from regular updatation.


for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f'step {iter} | Train loss {losses['train']:.4f} | val loss {losses['val']:.4f}')
    xb, yb= get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad()

    loss.backward()
    optimizer.step()

print(loss.item())
context = torch.zeros((1,1), dtype = torch.long, device = device)
print(decode(model.generate(idx = context, max_new_tokens=2500)[0].tolist()))


step 0 | Train loss 3.4022 | val loss 3.4443


[W929 15:03:48.853130713 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 100663296 bytes (free: 89915392, total: 3913940992).


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), 'res1.pt')

In [27]:
print(sum(p.numel() for p in model.parameters())/1e6, 'M Params')

10.759707 M Params


In [28]:
model.parameters

<bound method Module.parameters of BigramModel(
  (token_embedding): Embedding(27, 384)
  (positional_embedding): Embedding(256, 384)
  (blocks): Sequential(
    (0): Block(
      (sa_head): MultiHeadAttention(
        (heads): ModuleList(
          (0-5): 6 x Head(
            (key): Linear(in_features=384, out_features=64, bias=False)
            (query): Linear(in_features=384, out_features=64, bias=False)
            (value): Linear(in_features=384, out_features=64, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffwd): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((384,), eps=